# Stage 2A — single-view MAE (control arm)

Standard masked autoencoder pretraining on brain-MRI slices (75% masking, normalized-pixel targets). This is the CONTROL against which the multi-view twist is judged.

**Kaggle reality notes**
- Enable GPU + Internet (timm not needed here; Internet optional).
- `max_minutes=600` stops cleanly before the 12 h session limit; rerun the SAME
  notebook version (or a new version) and it **resumes** from `ckpt.pt`.
- Estimated wall time on T4 x2: single ≈ 3–5 h, cross ≈ 5–8 h for 60 epochs.


In [ ]:
import os, sys, subprocess
from pathlib import Path

IN_KAGGLE = Path("/kaggle").is_dir()
if IN_KAGGLE:
    repo = Path("mv-brainmae")
    if not repo.is_dir():
        # set your repo URL after pushing to GitHub
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/YOUR_GITHUB_USERNAME/mv-brainmae.git"],
                       check=True)
else:
    repo = Path("..")  # local: notebooks/ inside a clone
sys.path.insert(0, str(repo.resolve()))
os.chdir(repo)
print("repo:", repo.resolve())

import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
from mvbrainmae.utils import load_config
from mvbrainmae.pretrain import run_pretrain

cfg = load_config("configs/pretrain_single.yaml")
summary = run_pretrain(cfg)   # resumable: rerun after a session kill = continues

In [ ]:
# ---- VERIFICATION ----
import json
from pathlib import Path
import pandas as pd
recs = [json.loads(l) for l in open("outputs/pretrain_single/pretrain_single.jsonl")]
ep = pd.DataFrame([r for r in recs if r["event"] == "epoch"])
print(ep[["epoch", "loss"]].tail(8).to_string(index=False))
rc = pd.DataFrame([r for r in recs if r["event"] == "recon"])
print("val recon MSE trajectory:"); print(rc[["epoch", "val_recon_mse"]].to_string(index=False))
assert ep.loss.iloc[-1] < ep.loss.iloc[0], "loss did not decrease — check config"
assert Path("outputs/pretrain_single/encoder.pt").exists(), "encoder not saved"
print("VERIFIED: loss decreased and encoder checkpoint exists")

In [ ]:
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(9, 7))
latest = sorted(Path("outputs/pretrain_single").glob("recon_samples_e*.png"))[-1]
ax.imshow(Image.open(latest)); ax.axis("off")
ax.set_title(f"latest reconstruction grid — {latest.name}")
plt.tight_layout(); plt.show()